In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Aim: Identity Relationship

# i.e. $x = y$

![Identity Function](https://media.geeksforgeeks.org/wp-content/uploads/20250201153437076377/Identity-Function.webp)

# Generate Dataset $\mathcal{D} = (X, y)$

In [2]:
import torch

def create_identity_data(n):
    # n = total number of samples
    assert n % 2 == 0, "n must be even"

    half = n // 2

    # Negative and positive values
    X = torch.arange(-half, half, dtype=torch.float32).reshape(-1, 1)

    # Identity relationship: y = x
    y = X.clone()

    return X, y




In [3]:
X, y = create_identity_data(100)



In [4]:
from torch.utils.data import TensorDataset, DataLoader

# Train-test split
train_size = int(0.8 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

# Create datasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 80
Testing samples: 20


___

# Perceptron

$$
\hat{y} \leftarrow f_{\theta}(X)
$$

$$
\mathcal{L}(y,\hat{y}) \rightarrow \min
$$

$$
\displaystyle
J(\theta) = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

![Perceptron](https://media.geeksforgeeks.org/wp-content/uploads/20251209120638608023/bhu.webp)

In [5]:
import torch
import torch.nn as nn

class Perceptron(nn.Module):
    def __init__(self):
        super().__init__()

        self.linear = nn.Linear(1, 1)
        self.activation = nn.Identity()

    def forward(self, x):
        x = self.linear(x)
        x = self.activation(x)

        return x


model = Perceptron()

print(model)

Perceptron(
  (linear): Linear(in_features=1, out_features=1, bias=True)
  (activation): Identity()
)


___

# Parameter Description: $\theta$ (randomly initialised)

# Total Trainable Parameters = 2

# Randomly Initialised Parameter Values $\theta = \{W, b\} = \{ w_{1}^{1}, b_{1}^{1} \}$

In [6]:
weight = model.linear.weight.item()
bias = model.linear.bias.item()

print(f"Weight (w): {weight:.6f}")
print(f"Bias (b): {bias:.6f}")

Weight (w): 0.327111
Bias (b): -0.725591


___

# Training Loop

### Defining Hyperparameters: $\lambda$

In [7]:
criterion = nn.MSELoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001
)

epochs = 100

In [8]:


for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        # Forward pass
        y_pred = model(X_batch)

        # Loss
        loss = criterion(y_pred, y_batch)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [100/100], Loss: 0.0000


# Parameter Description: $\theta^{*}$ (trained)

In [9]:
weight = model.linear.weight.item()
bias = model.linear.bias.item()

print(f"Weight (w): {weight:.6f}")
print(f"Bias (b): {bias:.6f}")

Weight (w): 0.999915
Bias (b): -0.000956


In [10]:
import torch

# Random input
x = torch.tensor([[7.0]])

# Evaluation mode
model.eval()

with torch.no_grad():
    prediction = model(x)

print("Input:", x.item())
print("Predicted:", prediction.item())
print("Actual:", x.item())

Input: 7.0
Predicted: 6.998447895050049
Actual: 7.0


In [11]:
import torch
import torch.nn as nn

model.eval()

total_squared_error = 0.0
total_absolute_error = 0.0
total_samples = 0

all_predictions = []
all_targets = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        # Prediction
        y_pred = model(X_batch)

        # Store predictions and targets
        all_predictions.append(y_pred)
        all_targets.append(y_batch)

        # Errors
        total_squared_error += torch.sum((y_pred - y_batch) ** 2).item()
        total_absolute_error += torch.sum(torch.abs(y_pred - y_batch)).item()

        total_samples += y_batch.size(0)


# Combine all batches
predictions = torch.cat(all_predictions)
targets = torch.cat(all_targets)

# Metrics
mse = total_squared_error / total_samples
mae = total_absolute_error / total_samples
rmse = mse ** 0.5

# R² score
ss_res = torch.sum((targets - predictions) ** 2)
ss_tot = torch.sum((targets - targets.mean()) ** 2)

r2 = 1 - (ss_res / ss_tot)

print(f"MSE  : {mse:.6f}")
print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2.item():.6f}")

MSE  : 0.000019
MAE  : 0.004319
RMSE : 0.004347
R²   : 0.999999


In [12]:
tolerance = 0.1

correct = torch.abs(predictions - targets) <= tolerance

accuracy = correct.float().mean()

print(f"Accuracy (±{tolerance}): {accuracy.item() * 100:.2f}%")

Accuracy (±0.1): 100.00%
